# EEG Intersubject Synchrony — Per-Frequency-Band Analysis

This notebook mirrors `mean_variance_raw.ipynb` but applies the same analysis
independently to each EEG frequency band after bandpass-filtering the z-scored data.

**Frequency bands analysed:**
| Band  | Range |
|-------|-------|
| Delta | 1–4 Hz |
| Theta | 4–8 Hz |
| Alpha | 8–13 Hz |
| Beta  | 13–30 Hz |
| Gamma | 30–70 Hz |

**Sections:**
1. Intersubject mean & variance time series per band
2. Variance distribution per band
3. Inter-subject correlation (ISC) matrix per band
4. Windowed synchrony analysis per band

In [ ]:
import sys
import os

sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from src.analysis.summary import EEGSummarizedAnalyzer
from src.definitions.fields import (
    ExperimentNames,
    CoordinateSystems,
    ConditionVariants,
    MusicTypeVariants,
    ExclusionCategories,
    SingleDataMetadata,
    FrequencyBandNames,
)
from src.definitions.constants import ProjectPaths
from src.analysis.isc import (
    compute_mean_variance,
    compute_sliding_window_mean_variance,
)
from src.visualization.isc_plots import (
    plot_mean_variance_distribution,
    plot_sliding_window_mean_variance,
    plot_band_mean_variance_distributions,
    plot_band_sliding_window_mean_variance,
    print_data_overview,
)
from scripts.analysis_common import (
    load_analyzers,
    analyzers_to_datasets,
    run_isc_workflow,
    run_mean_variance_workflow,
)

%matplotlib inline

In [ ]:
# ── Experiment configuration ──────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL, MusicTypeVariants.PSYTRANCE]
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]

# Synchrony detection: windows whose mean variance is below this percentile
# of the band's own var_t distribution are marked as synchrony candidates
SYNC_PERCENTILE = 10

# Windowed analysis window length in seconds
WINDOW_DURATION_SEC = 2.0

# Set to True to reload from .fif files and resample/stack; False = load cached .npy
process_and_save_data = False

In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    process_and_save_data,
    normalize_data=True,
)
datasets = analyzers_to_datasets(analyzers)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pandas as pd
from matplotlib.patches import Patch

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)

# ── Dataset selection ─────────────────────────────────────────────────────
# Change LABEL to switch between music types
LABEL = MusicTypeVariants.CLASSICAL.value
# LABEL = MusicTypeVariants.PSYTRANCE.value

ad = datasets[LABEL]
data = ad.data        # (n_subjects, n_channels, n_times)
sfreq = ad.sfreq

n_subjects, n_channels, n_times = data.shape
time = np.arange(n_times) / sfreq   # seconds

print(f"Dataset : {LABEL}")
print(f"Shape   : {data.shape}  (subjects × channels × time points)")
print(f"Duration: {n_times / sfreq:.1f} s  @  {sfreq} Hz\n")

# ── Frequency band definitions ────────────────────────────────────────────
FREQUENCY_BANDS = {
    FrequencyBandNames.DELTA.value: (1.0, 4.0),
    FrequencyBandNames.THETA.value: (4.0, 8.0),
    FrequencyBandNames.ALPHA.value: (8.0, 13.0),
    FrequencyBandNames.BETA.value:  (13.0, 30.0),
    FrequencyBandNames.GAMMA.value: (30.0, 70.0),
}
BAND_NAMES = list(FREQUENCY_BANDS.keys())

# Assign a distinct colour to each band (used throughout the notebook)
BAND_COLORS = dict(zip(BAND_NAMES, sns.color_palette("tab10", n_colors=len(BAND_NAMES))))

# ── Bandpass-filter the data into each frequency band ─────────────────────
# Each entry is an AnalysisData with .data of shape (n_subjects, n_channels, n_times)
print("Filtering data to each frequency band...")
band_data = {}
for band, (l_freq, h_freq) in FREQUENCY_BANDS.items():
    band_data[band] = ad.filter_to_band(l_freq, h_freq)
    print(f"  {band:6s} ({l_freq:.0f}–{h_freq:.0f} Hz): {band_data[band].data.shape}")

## Section 1 — Intersubject Mean & Variance per Time Frame (per Band)

At each time point within each frequency band:

- **Intersubject mean** — mean across subjects, averaged over channels → shared response in that band
- **Intersubject variance** — variance *across subjects*, averaged over channels  
  → low = participants are in sync within that band
- **Per-subject traces** — each subject's channel-average per band

A 5-row figure shows mean signal (left column) and intersubject variance (right column) for every band.
The green dashed line marks the synchrony threshold (10th percentile of variance = low-variance moments).

In [ ]:
# ── Pre-compute intersubject statistics for each band ─────────────────────
# band_stats[band]: dict with inter_var, mean_t, var_t, std_t, mean_over_ch
band_stats = {}
for band, bd in band_data.items():
    bdata = bd.data   # (n_subjects, n_channels, n_times)
    inter_var_b  = bdata.var(axis=0)              # (n_channels, n_times)
    inter_mean_b = bdata.mean(axis=0)             # (n_channels, n_times)
    mean_t_b     = inter_mean_b.mean(axis=0)      # (n_times,)
    var_t_b      = inter_var_b.mean(axis=0)       # (n_times,)
    band_stats[band] = {
        "inter_var":   inter_var_b,
        "mean_t":      mean_t_b,
        "var_t":       var_t_b,
        "std_t":       np.sqrt(var_t_b),
        "mean_over_ch": bdata.mean(axis=1),       # (n_subjects, n_times)
    }

n_bands = len(BAND_NAMES)

# ── Two-column multi-row figure: mean signal (left) + variance (right) ────
fig, axes = plt.subplots(
    n_bands, 2,
    figsize=(16, 3.5 * n_bands),
    sharex=True,
)

for row, band in enumerate(BAND_NAMES):
    l_freq, h_freq = FREQUENCY_BANDS[band]
    st = band_stats[band]
    c  = BAND_COLORS[band]

    # --- Left panel: mean signal with per-subject traces ---
    ax = axes[row, 0]
    for s in range(n_subjects):
        ax.plot(time, st["mean_over_ch"][s], color=c, alpha=0.15, linewidth=0.5)
    ax.plot(time, st["mean_t"], color="black", linewidth=1.5, label="Group mean")
    ax.fill_between(
        time, st["mean_t"] - st["std_t"], st["mean_t"] + st["std_t"],
        color=c, alpha=0.25, label="±1 SD (intersubject)",
    )
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.6)
    ax.set_ylabel("Signal (z-score)", fontsize=8)
    if row == 0:
        ax.set_title("Mean signal per band", fontsize=10, fontweight="bold")
    ax.text(
        0.01, 0.97, f"{band.upper()} ({l_freq:.0f}–{h_freq:.0f} Hz)",
        transform=ax.transAxes, va="top", fontsize=9, fontweight="bold", color=c,
    )
    ax.legend(frameon=False, fontsize=7, loc="upper right")

    # --- Right panel: intersubject variance ---
    ax = axes[row, 1]
    ax.plot(time, st["var_t"], color=c, linewidth=1.2, alpha=0.7,
            label="Mean intersubject variance")
    ax.fill_between(time, 0, st["var_t"], color=c, alpha=0.18)
    ax.axhline(st["var_t"].mean(), color="gray", linestyle="--", linewidth=0.8,
               label=f"Grand mean ({st['var_t'].mean():.3f})")
    sync_thr = np.percentile(st["var_t"], SYNC_PERCENTILE)
    ax.axhline(sync_thr, color="#5cb85c", linestyle=":", linewidth=1.1,
               label=f"Sync thr — {SYNC_PERCENTILE}th pct ({sync_thr:.3f})")
    ax.set_ylabel("Variance", fontsize=8)
    if row == 0:
        ax.set_title(
            "Intersubject variance  (low = in sync)", fontsize=10, fontweight="bold"
        )
    ax.legend(frameon=False, fontsize=7, loc="upper right")

axes[-1, 0].set_xlabel("Time (s)")
axes[-1, 1].set_xlabel("Time (s)")

fig.suptitle(
    f"[{LABEL}]  Intersubject mean signal & variance per frequency band",
    fontsize=12, y=1.005,
)
sns.despine(fig=fig, left=False, bottom=False)
fig.tight_layout()
plt.show()

## Section 2 — Distribution of Intersubject Variance per Band

Histogram of intersubject variance values across all `(channel × time)` samples for each band,
clipped at the 99th percentile.

Different bands have inherently different absolute variance magnitudes; looking at each band
separately reveals the shape of the synchrony distribution within that spectral range.

In [ ]:
PLOT_PCT = 99    # clip at this percentile to suppress extreme outliers

ncols = 2
nrows = (n_bands + ncols - 1) // ncols   # ceiling division → 3 rows for 5 bands

fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.5 * nrows))
axes_flat = axes.ravel()

for idx, band in enumerate(BAND_NAMES):
    l_freq, h_freq = FREQUENCY_BANDS[band]
    st   = band_stats[band]
    c    = BAND_COLORS[band]

    all_var  = st["inter_var"].ravel()
    p_clip   = np.percentile(all_var, PLOT_PCT)
    n_out    = int((all_var > p_clip).sum())
    frac_out = 100.0 * n_out / all_var.size
    clipped  = all_var[all_var <= p_clip]

    ax = axes_flat[idx]
    counts, bin_edges = np.histogram(clipped, bins=80)
    pct_vals    = counts / all_var.size * 100
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    bw          = bin_edges[1] - bin_edges[0]

    ax.bar(bin_centers, pct_vals, width=bw * 0.95, color=c, alpha=0.85, edgecolor="none")
    ax.axvline(np.median(all_var), color=".2", linestyle="--", linewidth=1.1,
               label=f"Median = {np.median(all_var):.3f}")
    ax.axvline(p_clip, color="crimson", linestyle=":", linewidth=1.3,
               label=f"{PLOT_PCT}th pct = {p_clip:.3f}")
    ax.set_title(
        f"{band.upper()} ({l_freq:.0f}–{h_freq:.0f} Hz)\n"
        f"⚠ {n_out:,} outliers ({frac_out:.1f}%) not shown",
        fontsize=9,
    )
    ax.set_xlabel("Intersubject variance", fontsize=8)
    ax.set_ylabel("% of all channel × time samples", fontsize=8)
    ax.legend(frameon=False, fontsize=7)

# Hide unused axes (last cell if n_bands is odd)
for idx in range(n_bands, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle(
    f"[{LABEL}]  Variance distribution per frequency band (clipped at {PLOT_PCT}th pct)",
    fontsize=11,
)
sns.despine(fig=fig)
fig.tight_layout()
plt.show()

# ── Print summary statistics ──────────────────────────────────────────────
print(f"\nVariance summary per band [{LABEL}]")
print(f"{'Band':<8} {'Mean':>8} {'Median':>8} {'Std':>8} {'99th pct':>10}")
print("-" * 48)
for band in BAND_NAMES:
    v = band_stats[band]["inter_var"].ravel()
    print(f"{band:<8} {v.mean():>8.4f} {np.median(v):>8.4f} {v.std():>8.4f} {np.percentile(v,99):>10.4f}")

## Section 3 — Inter-subject Correlation (ISC) Matrix per Band

For every pair of subjects $(i, j)$ and each frequency band, the **mean Pearson correlation**
across channels of the band-filtered z-scored signals is computed:

$$\text{ISC}(i,j) = \frac{1}{C} \sum_{c=1}^{C} \frac{1}{T} \sum_{t=1}^{T} x^{(i)}_{c,t} \cdot x^{(j)}_{c,t}$$

For z-scored data ($\mu=0, \sigma=1$) this equals the mean Pearson correlation across channels.

- **High value** → subjects follow similar band dynamics (in sync)
- **Low / negative** → subjects diverge in that band

In [ ]:
n_subj        = n_subjects
subject_labels = [f"S{s + 1}" for s in range(n_subj)]

# ── Compute ISC matrix for every band ────────────────────────────────────
isc_matrices = {}
for band in BAND_NAMES:
    bdata = band_data[band].data   # (n_subjects, n_channels, n_times)
    mat   = np.zeros((n_subj, n_subj))
    for i in range(n_subj):
        for j in range(n_subj):
            # mean Pearson correlation: mean_ch( mean_t( x_i * x_j ) )
            mat[i, j] = np.mean((bdata[i] * bdata[j]).mean(axis=1))
    isc_matrices[band] = mat

# ── Plot: one heatmap per band ─────────────────────────────────────────────
ncols = 2
nrows = (n_bands + ncols - 1) // ncols

cell_size = max(5, n_subj * 0.45)
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(ncols * max(6, n_subj * 0.5), nrows * cell_size),
)
axes_flat = axes.ravel()

for idx, band in enumerate(BAND_NAMES):
    l_freq, h_freq = FREQUENCY_BANDS[band]
    mat  = isc_matrices[band]
    tril = np.tril_indices(n_subj, k=-1)
    pv   = mat[tril]

    ax = axes_flat[idx]
    sns.heatmap(
        mat, ax=ax,
        vmin=0, vmax=1,
        cmap="Blues",
        annot=(n_subj <= 20), fmt=".2f" if n_subj <= 20 else "",
        annot_kws={"size": 6} if n_subj <= 20 else {},
        linewidths=0.3 if n_subj <= 20 else 0,
        square=True,
        xticklabels=subject_labels,
        yticklabels=subject_labels,
        cbar_kws={"label": "Mean ISC", "shrink": 0.8},
    )
    ax.set_title(
        f"{band.upper()}  ({l_freq:.0f}–{h_freq:.0f} Hz)\n"
        f"ISC — min: {pv.min():.3f}  max: {pv.max():.3f}  median: {np.median(pv):.3f}",
        fontsize=9,
    )
    ax.set_xlabel("Subject", fontsize=8)
    ax.set_ylabel("Subject", fontsize=8)
    ax.tick_params(labelsize=7)

for idx in range(n_bands, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle(
    f"[{LABEL}]  Inter-subject correlation matrices per frequency band\n"
    f"(mean Pearson ISC of z-scored band-filtered signals — high = in sync)",
    fontsize=11,
)
fig.tight_layout()
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────
print(f"\nISC summary per band [{LABEL}]  (lower triangle, excl. diagonal)")
print(f"{'Band':<8} {'Min':>8} {'Max':>8} {'Median':>8} {'Mean':>8}")
print("-" * 42)
for band in BAND_NAMES:
    tril = np.tril_indices(n_subj, k=-1)
    pv   = isc_matrices[band][tril]
    print(f"{band:<8} {pv.min():>8.4f} {pv.max():>8.4f} {np.median(pv):>8.4f} {pv.mean():>8.4f}")

# ── Most / least synchronous pairs per band ───────────────────────────────
for band in BAND_NAMES:
    l_freq, h_freq = FREQUENCY_BANDS[band]
    tril     = np.tril_indices(n_subj, k=-1)
    pv       = isc_matrices[band][tril]
    order    = np.argsort(pv)
    print(f"\n  {band.upper()} ({l_freq:.0f}–{h_freq:.0f} Hz) — top synchronous pairs:")
    for k in order[-3:][::-1]:
        i, j = tril[0][k], tril[1][k]
        print(f"    S{i+1} – S{j+1}: {pv[k]:.4f}")
    print(f"  Least synchronous:")
    for k in order[:3]:
        i, j = tril[0][k], tril[1][k]
        print(f"    S{i+1} – S{j+1}: {pv[k]:.4f}")

## Section 4 — Windowed Synchrony Analysis per Band

The recording is divided into non-overlapping windows of `WINDOW_DURATION_SEC` seconds.
For each band and each window the mean intersubject variance is computed.

Windows whose mean variance falls below the **`SYNC_PERCENTILE`th percentile** of that band's
own `var_t` distribution are labelled *synchrony candidates* (green).

Two figures are produced per band:

1. **Summary figure** (all bands in one multi-row plot) — windowed variance + sync spans  
2. **Per-band detailed figure** — continuous variance overlay + windowed step + per-electrode heatmap

In [ ]:
muted    = sns.color_palette("muted")
C_ORANGE = muted[1]
C_GREEN  = "#5cb85c"
C_RED    = muted[2]

win_samples = int(WINDOW_DURATION_SEC * sfreq)
n_windows   = n_times // win_samples

# ── Build per-band windowed statistics ────────────────────────────────────
band_win_stats = {}
for band in BAND_NAMES:
    st        = band_stats[band]
    var_t_b   = st["var_t"]
    sync_thr  = np.percentile(var_t_b, SYNC_PERCENTILE)

    records = []
    for w in range(n_windows):
        sl = slice(w * win_samples, (w + 1) * win_samples)
        records.append({
            "window":        w + 1,
            "center":        time[w * win_samples + win_samples // 2],
            "t_start":       time[w * win_samples],
            "t_end":         time[min((w + 1) * win_samples - 1, n_times - 1)],
            "mean_variance": var_t_b[sl].mean(),
        })

    df_wins = pd.DataFrame(records)
    df_wins["sync_candidate"] = df_wins["mean_variance"] < sync_thr

    _pad              = n_times - n_windows * win_samples
    win_mean_var_step = np.pad(
        np.repeat(df_wins["mean_variance"].values, win_samples),
        (0, _pad), mode="edge",
    )

    band_win_stats[band] = {
        "df_wins":          df_wins,
        "sync_threshold":   sync_thr,
        "win_mean_var_step": win_mean_var_step,
        "is_sync_win":      df_wins["sync_candidate"].values,
    }

# ════════════════════════════════════════════════════════════════════════════
# Summary figure — all bands in one multi-row plot
# ════════════════════════════════════════════════════════════════════════════
fig_s, axes_s = plt.subplots(
    n_bands, 1,
    figsize=(16, 2.8 * n_bands),
    sharex=True,
)

for row, band in enumerate(BAND_NAMES):
    l_freq, h_freq = FREQUENCY_BANDS[band]
    st  = band_stats[band]
    ws  = band_win_stats[band]
    c   = BAND_COLORS[band]
    ax  = axes_s[row]

    ds  = max(1, n_times // 8000)
    t_ds = time[::ds]

    ax.fill_between(t_ds, 0, st["var_t"][::ds], color=c, alpha=0.15)
    ax.plot(t_ds, st["var_t"][::ds], color=c, linewidth=0.8, alpha=0.6)
    ax.step(time, ws["win_mean_var_step"], where="post",
            color=c, linewidth=2.0, label=f"{WINDOW_DURATION_SEC:.1f} s windowed mean")
    ax.axhline(ws["sync_threshold"], color=C_GREEN, linestyle="--", linewidth=1.0,
               label=f"Sync thr ({SYNC_PERCENTILE}th pct = {ws['sync_threshold']:.3f})")

    first_s = True
    for w in np.where(ws["is_sync_win"])[0]:
        t_s = time[w * win_samples]
        t_e = time[min((w + 1) * win_samples - 1, n_times - 1)]
        ax.axvspan(t_s, t_e, color=C_GREEN, alpha=0.22,
                   label="Sync candidate" if first_s else "_nolegend_")
        first_s = False

    n_sync = ws["is_sync_win"].sum()
    ax.set_ylabel("Variance", fontsize=8)
    ax.text(
        0.01, 0.97,
        f"{band.upper()} ({l_freq:.0f}–{h_freq:.0f} Hz)  — {n_sync}/{n_windows} sync windows",
        transform=ax.transAxes, va="top", fontsize=9, fontweight="bold", color=c,
    )
    ax.legend(frameon=False, fontsize=7, loc="upper right")

axes_s[-1].set_xlabel("Time (s)")
fig_s.suptitle(
    f"[{LABEL}]  Windowed intersubject variance per band  (window = {WINDOW_DURATION_SEC:.1f} s)",
    fontsize=11,
)
sns.despine(fig=fig_s, left=False, bottom=False)
fig_s.tight_layout()
plt.show()

# ════════════════════════════════════════════════════════════════════════════
# Per-band detailed figure — variance overlay + electrode heatmap
# ════════════════════════════════════════════════════════════════════════════
for band in BAND_NAMES:
    l_freq, h_freq = FREQUENCY_BANDS[band]
    st  = band_stats[band]
    ws  = band_win_stats[band]
    c   = BAND_COLORS[band]

    ds   = max(1, n_times // 8000)
    t_ds = time[::ds]
    df_var = pd.DataFrame({
        "time":     t_ds,
        "variance": st["var_t"][::ds],
        "windowed": ws["win_mean_var_step"][::ds],
    })

    fig2 = plt.figure(figsize=(15, 7))
    gs2  = gridspec.GridSpec(
        2, 2,
        height_ratios=[2, 3],
        width_ratios=[30, 1],
        hspace=0.15, wspace=0.05,
        figure=fig2,
    )
    ax_v  = fig2.add_subplot(gs2[0, 0])
    ax_hm = fig2.add_subplot(gs2[1, 0], sharex=ax_v)
    cax   = fig2.add_subplot(gs2[1, 1])

    # --- Variance overlay ---
    ax_v.plot(df_var["time"], df_var["variance"],
              color=c, linewidth=1.0, alpha=0.55, label="Continuous intersubject variance")
    ax_v.fill_between(df_var["time"], 0, df_var["variance"], color=c, alpha=0.12)
    ax_v.step(df_var["time"], df_var["windowed"], where="post",
              color=C_RED, linewidth=2.0, label=f"Windowed mean ({WINDOW_DURATION_SEC:.1f} s)")
    ax_v.axhline(ws["sync_threshold"], color=C_GREEN, linestyle="--", linewidth=1.1,
                 label=f"Sync threshold — {SYNC_PERCENTILE}th pct ({ws['sync_threshold']:.3f})")
    first_s = True
    for w in np.where(ws["is_sync_win"])[0]:
        t_s = time[w * win_samples]
        t_e = time[min((w + 1) * win_samples - 1, n_times - 1)]
        ax_v.axvspan(t_s, t_e, color=C_GREEN, alpha=0.22,
                     label="Sync candidate window" if first_s else "_nolegend_")
        first_s = False
    ax_v.set_ylabel("Intersubject variance")
    ax_v.set_title(
        f"[{LABEL}]  {band.upper()} ({l_freq:.0f}–{h_freq:.0f} Hz) — "
        f"variance overlay + electrode heatmap  (window = {WINDOW_DURATION_SEC:.1f} s)"
    )
    ax_v.legend(frameon=False, fontsize=9)
    plt.setp(ax_v.get_xticklabels(), visible=False)

    # --- Per-electrode variance heatmap ---
    ds_hm     = max(1, n_times // 2000)
    _time_hm  = time[::ds_hm]
    _var_hm   = st["inter_var"][:, ::ds_hm]
    _dt       = (_time_hm[1] - _time_hm[0]) if len(_time_hm) > 1 else 1.0 / sfreq
    _t_edges  = np.r_[_time_hm - _dt / 2, _time_hm[-1] + _dt / 2]
    _ch_edges = np.arange(n_channels + 1)

    pcm = ax_hm.pcolormesh(
        _t_edges, _ch_edges, _var_hm,
        cmap="YlOrRd",
        vmin=0,
        vmax=np.percentile(st["inter_var"], 98),
        rasterized=True,
        shading="flat",
    )
    ax_hm.set_ylim(n_channels, 0)   # electrode 0 at top
    fig2.colorbar(pcm, cax=cax, label="Intersubject variance")

    first_s = True
    for w in np.where(ws["is_sync_win"])[0]:
        t_s = time[w * win_samples]
        t_e = time[min((w + 1) * win_samples - 1, n_times - 1)]
        ax_hm.axvspan(t_s, t_e, color=C_GREEN, alpha=0.18,
                      label="Sync candidate window" if first_s else "_nolegend_")
        first_s = False
    ax_hm.set_xlabel("Time (s)")
    ax_hm.set_ylabel("Electrode (natural order)")

    sns.despine(fig=fig2, left=False, bottom=False)
    fig2.tight_layout()
    plt.show()

    # ── Print synchrony-candidate windows for this band ──────────────────
    sync_df = ws["df_wins"][ws["df_wins"]["sync_candidate"]][
        ["window", "t_start", "t_end", "mean_variance"]
    ]
    print(
        f"\n{band.upper()} ({l_freq:.0f}–{h_freq:.0f} Hz)"
        f"  |  Window = {WINDOW_DURATION_SEC:.1f} s"
        f"  |  Sync threshold = {ws['sync_threshold']:.4f}"
        f"  ({SYNC_PERCENTILE}th pct)"
    )
    if len(sync_df):
        print(f"  Synchrony-candidate windows ({len(sync_df)}):")
        print(sync_df.to_string(index=False, float_format="%.4f"))
    else:
        print("  No synchrony-candidate windows found.")